In [ ]:
# Task 1: Data Acquisition, Cleaning, and Preprocessing

## Objective

##The objective of this task is to acquire, inspect, clean, and preprocess an air-quality dataset using Python and Pandas. The workflow includes data-quality assessment, handling missing and invalid values, outlier detection, date-time preprocessing, validation, and export of the cleaned dataset.

In [ ]:
## 1. Data Acquisition and Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import openpyxl

In [ ]:
## 2. Initial Data Exploration

In [ ]:
# path to the raw dataset
file_path = "../../data/AirQualityUCI.csv"

# load the dataset
df = pd.read_csv(file_path, sep=';', decimal=',')

print("Dataset loaded successfully!")
print("Shape:", df.shape)
df.head()

In [ ]:
# Check column names
print("Columns:")
print(df.columns.tolist())

In [ ]:
# Check data types
print("Data Types:")
print(df.dtypes)

In [ ]:
## 3. Data Quality Assessment

In [ ]:
# Check dataset information
print("Data Information:")
df.info()

In [ ]:
print("Unnamed: 15 unique values:")
print(df["Unnamed: 15"].unique())

print("\nUnnamed: 16 unique values: ")
print(df["Unnamed: 16"].unique())

# Count completely empty rows
print("\nCompletely empty rows:", df.isna().all(axis=1).sum())

# Count -200 values by each column
print("\n -200 values by column:")
print((df == -200).sum())

In [ ]:
## 4. Missing-Value Analysis

In [ ]:
# Check duplicate values
duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

# Show a few duplicate row if any exist
if duplicate_count > 0:
    print("\nSample duplicate rows:")
    print(df[df.duplicated(keep=False)].head(10))

In [ ]:
# Calculate missing values represented by -200
missing_200 = (df == -200).sum()

# Calculate missing-value percentage
missing_percentage = (missing_200 / len(df)) * 100

missing_summary = pd.DataFrame({
    "Missing (-200) Count": missing_200,
    "Missing(%)": missing_percentage.round(2)
})
print(missing_summary)

In [ ]:
    # Statistical summary of numerical columns
df.describe().T

In [ ]:
# Check how many -200 values occur in each row
missing_per_row = (df == -200).sum(axis=1)

print("Rows containing missing (-200) count:")
print((missing_per_row > 0).sum())

print("\nRows containing more than 5 missing values:")
print((missing_per_row > 5).sum())

print("\nMaximum missing value per row:")
print(missing_per_row.max())

In [ ]:
# Distribution of number of missing values per row
print(missing_per_row.value_counts().sort_index())

In [ ]:
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

print("First date:", df["Date"].min())
print("Last date:", df["Date"].max())

print("\nNumber of unique dates:", df["Date"].nunique())

In [ ]:
# Rows with the highest number of missing measurements
top_missing_rows = missing_per_row.nlargest(10)

print(df.loc[top_missing_rows.index])

In [ ]:
# Number of -200 values in each row
missing_per_row = (df == -200).sum(axis=1)

print("Rows containing at least one -200:", (missing_per_row > 0).sum())
print("Rows containing more than 5 -200 values:", (missing_per_row > 5).sum())
print("Maximum -200 values in one row:", missing_per_row.max())

print("\nDistribution:")
print(missing_per_row.value_counts().sort_index())

In [ ]:
# Rows with 9 or more missing measurements
severe_missing = df.loc[missing_per_row >= 9, ["Date", "Time"]].copy()

print("Number of severely incomplete rows:", len(severe_missing))

print("\nFirst 20:")
print(severe_missing.head(20))

print("\nLast 20:")
print(severe_missing.tail(20))

In [ ]:
## 5. Data Cleaning and Missing-Value Treatment

In [ ]:
# Count severe missingness by date
severe_by_date = (
    df.loc[missing_per_row >= 9]
      .groupby("Date")
      .size()
      .sort_values(ascending=False)
)

print(severe_by_date.head(20))

In [ ]:
# Create a working copy of the raw dataset
df_clean = df.copy()

#1. Remove completely empty columns
df_clean = df_clean.dropna(axis=1, how="all")

#2. Remove completely empty rows
df_clean = df_clean.dropna(axis=0, how="all")

# Check the result
print("Shape after removing empty rows n columns", df_clean.shape)

In [ ]:
# Convert the dataset's missing-value marker to NaN
df_clean = df_clean.replace(-200, np.nan)

print("Missing values after conversion:")
print(df_clean.isna().sum())

In [ ]:
print("Missing percentange:")
print((df_clean.isna().mean()*100).round(2))

In [ ]:
# NMHC(GT) has extremely high missingness (~90%)
df_clean = df_clean.drop(columns=["NMHC(GT)"])

print("Shape after removing NMHC(GT):", df_clean.shape)

In [ ]:
missing_summary = pd.DataFrame({
    "Missing Count": df_clean.isna().sum(),
    "Missing Percentage": (df_clean.isna().mean() * 100).round(2)
})

print(missing_summary)

In [ ]:
# Numerical columns excluding Date and Time
numeric_cols = df_clean.select_dtypes(include=np.number).columns

# Fill missing numerical values with the median of each column
for col in numeric_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Check remaining missing values
print("Remaining missing values:")
print(df_clean.isna().sum())

In [ ]:
print("Final shape:", df_clean.shape)

print("\nMissing values:")
print(df_clean.isna().sum())

print("\nDuplicate rows:", df_clean.duplicated().sum())

print("\nData types:")
print(df_clean.dtypes)

In [ ]:
## 6. Outlier Detection

In [ ]:
print("\nRemaining -200 values:", (df_clean == -200).sum().sum())

In [ ]:
# Select numerical columns
numeric_cols = df_clean.select_dtypes(include=np.number).columns

# Calculate Q1, Q3 and IQR
Q1 = df_clean[numeric_cols].quantile(0.25)
Q3 = df_clean[numeric_cols].quantile(0.75)

IQR = Q3 - Q1

# Define lower and upper bounds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Count potential outliers in each column
outlier_count = (
    ((df_clean[numeric_cols] < lower_bound) |
     (df_clean[numeric_cols] > upper_bound))
    .sum()
)

outlier_percentage = (outlier_count / len(df_clean) * 100).round(2)

outlier_summary = pd.DataFrame({
    "Outlier Count": outlier_count,
    "Outlier Percentage": outlier_percentage
}).sort_values("Outlier Count", ascending=False)

print(outlier_summary)

In [ ]:
import matplotlib.pyplot as plt

# Boxplots for major pollutant variables
pollutants = ["CO(GT)", "NOx(GT)", "NO2(GT)", "C6H6(GT)"]

for col in pollutants:
    plt.figure(figsize=(8, 5))
    plt.boxplot(df_clean[col])
    plt.title(f"Boxplot of {col}")
    plt.ylabel(col)
    plt.show()

In [ ]:
## 7. Date and Time Preprocessing

In [ ]:
# Convert Date + Time into a proper datetime
df_clean["Datetime"] = pd.to_datetime(
    df_clean["Date"].astype(str) + " " + df_clean["Time"].astype(str),
    format="%Y-%m-%d %H.%M.%S",
    errors="coerce"
)

print(df_clean[["Date", "Time", "Datetime"]].head())
print("\nInvalid datetime values:", df_clean["Datetime"].isna().sum())

In [ ]:
# Remove the original Date and Time columns
df_clean = df_clean.drop(columns=["Date", "Time"])

# Put Datetime as the first column
cols = ["Datetime"] + [col for col in df_clean.columns if col != "Datetime"]
df_clean = df_clean[cols]

print("Final shape:", df_clean.shape)
print("\nColumns:")
print(df_clean.columns.tolist())

print("\nData types:")
print(df_clean.dtypes)

In [ ]:
## 8. Final Data Validation

In [ ]:
print("Missing values:", df_clean.isna().sum().sum())
print("Remaining -200 values:", (df_clean == -200).sum().sum())
print("Duplicate rows:", df_clean.duplicated().sum())
print("Invalid Datetime:", df_clean["Datetime"].isna().sum())

In [ ]:
## 9. Exporting the Cleaned Dataset

In [ ]:
import os

os.makedirs("../outputs", exist_ok=True)

output_path = "../outputs/air_quality_cleaned.csv"

df_clean.to_csv(output_path, index=False)

print("Cleaned dataset saved successfully!")
print("Path:", output_path)

In [ ]:
import os

print("File exists:", os.path.exists(output_path))
print("File size:", os.path.getsize(output_path), "bytes")

In [ ]:
## 10. Conclusion

## The dataset was successfully inspected, cleaned, and validated. Invalid -200 placeholders were converted to missing values, highly incomplete data was addressed, statistical outliers were identified and retained where they could represent genuine environmental observations, and the separate date and time fields were converted into a unified datetime column. The resulting dataset is ready for further analysis.

In [ ]:
import os

output_path = "../outputs/air_quality_cleaned.csv"

print("File exists:", os.path.exists(output_path))

if os.path.exists(output_path):
    print("File size:", os.path.getsize(output_path), "bytes")